# 🐘 AI Elephant Detection — Smart Traffic Light System
> **Google Colab Notebook**  
> Uses Roboflow for elephant detection + distance-based traffic light logic  
> Designed to run with image uploads (webcam not natively supported in Colab)

---
### 🗺️ System Architecture
```
📷 Camera/Image  →  🤖 Roboflow Model  →  📏 Distance Estimation
                                               ↓
                          >40 m  →  🟢 GREEN  (safe)
                        20–40 m  →  🟡 YELLOW (warning)
                          <20 m  →  🔴 RED    (danger / stop)
                                               ↓
                                     📡 IoT Alert Sent
```


## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install -q roboflow inference-sdk opencv-python-headless Pillow requests streamlit pyngrok
print('✅ All packages installed!')

## 🔑 Step 2 — Configure Roboflow API Credentials

In [ ]:
# ─── FILL IN YOUR DETAILS ───────────────────────────────────────
ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'    # from roboflow.com → Settings → API
WORKSPACE_ID     = 'YOUR_WORKSPACE_ID'   # e.g.  'najwa-lab'
PROJECT_ID       = 'elephant-detection'  # your Roboflow project slug
MODEL_VERSION    = 1                     # version number of the trained model

# Distance thresholds (metres)
YELLOW_THRESHOLD = 40
RED_THRESHOLD    = 20

# IoT endpoint (optional – leave blank to skip)
IOT_ENDPOINT = ''   # e.g. 'http://192.168.1.100/api/traffic'

print('✅ Config ready')

## 🤖 Step 3 — Load the Roboflow Model

In [ ]:
from inference_sdk import InferenceHTTPClient

client = InferenceHTTPClient(
    api_url='https://detect.roboflow.com',
    api_key=ROBOFLOW_API_KEY
)
MODEL_ID = f'{PROJECT_ID}/{MODEL_VERSION}'
print(f'✅ Roboflow client ready — model: {MODEL_ID}')

## 📏 Step 4 — Distance Estimation & Traffic Light Logic

In [ ]:
import math

KNOWN_ELEPHANT_HEIGHT_M = 2.8   # average elephant height (metres)
FOCAL_LENGTH_PX         = 700   # camera focal length in pixels (calibrate to your camera)

def estimate_distance(bbox_height_px, frame_height_px=720):
    """
    Pinhole camera model:
        distance = (known_real_height * focal_length) / pixel_height
    """
    if bbox_height_px <= 0:
        return 999.0
    return round((KNOWN_ELEPHANT_HEIGHT_M * FOCAL_LENGTH_PX) / bbox_height_px, 2)


def get_traffic_status(distance_m):
    if distance_m <= RED_THRESHOLD:
        return 'RED',    '🔴', f'🚨 DANGER — Elephant at {distance_m} m — STOP!'
    if distance_m <= YELLOW_THRESHOLD:
        return 'YELLOW', '🟡', f'⚠️  WARNING — Elephant at {distance_m} m — Slow down!'
    return     'GREEN',  '🟢', f'✅  Clear — nearest elephant at {distance_m} m'


# Quick sanity test
for d in [15, 30, 55]:
    s, e, m = get_traffic_status(d)
    print(f'{e} {s:6}  →  {m}')

## 📡 Step 5 — IoT Alert Function

In [ ]:
import requests, time, json

def send_iot_alert(status, distance_m, endpoint=IOT_ENDPOINT):
    payload = {
        'status':     status,
        'signal':     {'GREEN': 0, 'YELLOW': 1, 'RED': 2}[status],
        'distance_m': distance_m,
        'timestamp':  time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'alert':      status in ('RED', 'YELLOW'),
    }
    print(f'📡 IoT Payload → {json.dumps(payload, indent=2)}')

    if endpoint:
        try:
            r = requests.post(endpoint, json=payload, timeout=3)
            print(f'   HTTP {r.status_code}: {r.text[:120]}')
        except Exception as e:
            print(f'   [Simulation mode] Device not reachable: {e}')
    else:
        print('   [Simulation mode] No endpoint configured.')
    return payload

print('✅ IoT alert function ready')

## 🖼️ Step 6 — Annotate + Display Function

In [ ]:
import cv2
import numpy as np
from PIL import Image
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COLOUR_MAP = {
    'GREEN':  (0, 200, 80),
    'YELLOW': (0, 200, 255),
    'RED':    (60, 60, 255),
}

def draw_detections(frame_bgr, predictions, frame_h=720, conf_thresh=0.45):
    closest = 999.0
    frame   = frame_bgr.copy()

    for pred in predictions:
        if pred.get('confidence', 0) < conf_thresh:
            continue
        x   = int(pred['x'] - pred['width']  / 2)
        y   = int(pred['y'] - pred['height'] / 2)
        w   = int(pred['width'])
        h   = int(pred['height'])
        conf = pred.get('confidence', 0)

        dist   = estimate_distance(h, frame_h)
        closest = min(closest, dist)
        status, _, _ = get_traffic_status(dist)
        colour = COLOUR_MAP[status]

        cv2.rectangle(frame, (x, y), (x + w, y + h), colour, 3)
        label = f"Elephant | {conf:.0%} | {dist:.1f} m"
        (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
        cv2.rectangle(frame, (x, y - lh - 12), (x + lw + 8, y), colour, -1)
        cv2.putText(frame, label, (x + 4, y - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

    return frame, closest


def show_result(frame_bgr, status, emoji, message, distance):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    colour_bg = {'GREEN': '#1a3a1a', 'YELLOW': '#3a3a00', 'RED': '#3a0000'}[status]
    colour_fg = {'GREEN': '#00C851', 'YELLOW': '#FFD700', 'RED': '#FF4444'}[status]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6),
                             gridspec_kw={'width_ratios': [3, 1]})
    fig.patch.set_facecolor('#0e1117')

    # Camera feed
    axes[0].imshow(frame_rgb)
    axes[0].axis('off')
    axes[0].set_title('📷 Live Detection Feed', color='white', fontsize=13)

    # Traffic light panel
    axes[1].set_facecolor(colour_bg)
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[1].axis('off')

    axes[1].text(0.5, 0.78, emoji,         ha='center', va='center',
                 fontsize=72, transform=axes[1].transAxes)
    axes[1].text(0.5, 0.60, status,        ha='center', va='center',
                 fontsize=28, fontweight='bold', color=colour_fg,
                 transform=axes[1].transAxes)
    axes[1].text(0.5, 0.48, f'{distance:.1f} m',
                 ha='center', va='center', fontsize=20, color='white',
                 transform=axes[1].transAxes)
    axes[1].text(0.5, 0.30, message,       ha='center', va='center',
                 fontsize=9, color='#cccccc', wrap=True,
                 transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()

print('✅ Visualization functions ready')

## 🧪 Step 7 — Test with a Single Image Upload
> Upload any elephant photo to test the full pipeline.

In [ ]:
from google.colab import files
import io

print('Upload an image (JPG/PNG) ...')
uploaded = files.upload()

for filename, data in uploaded.items():
    print(f'\n📁 Processing: {filename}')

    # Load image
    pil_img    = Image.open(io.BytesIO(data)).convert('RGB')
    frame_bgr  = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    frame_h    = frame_bgr.shape[0]

    # Run inference
    result     = client.infer(pil_img, model_id=MODEL_ID)
    predictions = result.get('predictions', [])
    print(f'🔍 Roboflow found {len(predictions)} detection(s)')

    if predictions:
        frame_annotated, closest = draw_detections(frame_bgr, predictions, frame_h)
    else:
        frame_annotated = frame_bgr
        closest = 999.0
        print('   No elephants detected in this image.')

    status, emoji, message = get_traffic_status(closest)
    print(f'\n{emoji} TRAFFIC STATUS: {status}')
    print(f'   Distance : {closest} m')
    print(f'   Message  : {message}')

    # Send IoT alert
    send_iot_alert(status, closest)

    # Show result
    show_result(frame_annotated, status, emoji, message, closest)

## 📹 Step 8 — Process a Video File
> Upload an MP4/AVI video to run frame-by-frame detection.

In [ ]:
from google.colab import files
from IPython.display import Video
import os

print('Upload a video file (MP4 recommended) ...')
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

cap        = cv2.VideoCapture(video_path)
fps        = cap.get(cv2.CAP_PROP_FPS) or 25
width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out_path   = 'output_annotated.mp4'

fourcc     = cv2.VideoWriter_fourcc(*'mp4v')
out        = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

frame_idx  = 0
infer_every = 5   # run inference every N frames for speed
last_preds  = []
prev_status = 'GREEN'

print(f'⚙️  Processing video: {video_path}  ({width}x{height} @ {fps:.1f} fps)...')

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1
    if frame_idx % infer_every == 0:
        pil_img    = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        result     = client.infer(pil_img, model_id=MODEL_ID)
        last_preds = result.get('predictions', [])

    if last_preds:
        frame, closest = draw_detections(frame, last_preds, height)
        status, _, _   = get_traffic_status(closest)
        if status != prev_status:
            send_iot_alert(status, closest)
            prev_status = status
    else:
        closest = 999.0

    # Overlay status
    col = COLOUR_MAP[prev_status]
    cv2.putText(frame, f'STATUS: {prev_status}  |  {closest:.1f} m',
                (10, 36), cv2.FONT_HERSHEY_SIMPLEX, 1.1, col, 3)
    out.write(frame)

    if frame_idx % 50 == 0:
        print(f'   Frame {frame_idx}  |  Status: {prev_status}  |  Distance: {closest:.1f} m')

cap.release()
out.release()

print(f'\n✅ Done! Annotated video saved → {out_path}')
Video(out_path, embed=True)

## 🌐 Step 9 — Launch Streamlit Dashboard in Colab via ngrok
> This tunnels the Streamlit dashboard to a public URL you can open in your browser.

In [ ]:
# ── FILL IN your ngrok authtoken from https://dashboard.ngrok.com ──
NGROK_TOKEN = 'YOUR_NGROK_AUTH_TOKEN'

!pip install -q pyngrok
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()  # kill any previous tunnels

# Write app files to Colab runtime
# (You can also upload app.py and iot_alert.py using the Files panel)
print('Starting Streamlit ...')
!streamlit run app.py --server.port 8501 &>/dev/null &

import time; time.sleep(3)
tunnel = ngrok.connect(8501)
print(f'\n🌐 Dashboard live at: {tunnel.public_url}')

---
## 📝 Notes

| Parameter | Default | How to change |
|---|---|---|
| Yellow threshold | 40 m | Change `YELLOW_THRESHOLD` cell above |
| Red threshold | 20 m | Change `RED_THRESHOLD` cell above |
| Focal length | 700 px | Calibrate using a known distance + ruler |
| Elephant height | 2.8 m | Adjust for Asian (2.5 m) vs African (3.3 m) |
| IoT endpoint | blank | Set `IOT_ENDPOINT` to your device URL |

### 📷 Calibrating Focal Length
1. Place an elephant (or object of known height) at exactly **D** metres from the camera.
2. Measure its bounding box height **P** pixels in the image.
3. `FOCAL_LENGTH_PX = (P * D) / KNOWN_HEIGHT_M`

### 🔑 Getting a Roboflow API Key
1. Sign up at [roboflow.com](https://roboflow.com)
2. Create or fork an **elephant detection** project (search Universe)
3. Train / select a model version
4. Go to Settings → API Key
